# F3_01 · Núcleo orientado a objetos (Fase 3)

**Curso:** MCDI500 · Programación para la Ciencia de Datos · **Grupo 2**
**Proyecto:** Colecistectomía en grupos de riesgo de cáncer de vesícula, La Araucanía 2022

## Objetivo

Este notebook toma la transformación de la Fase 2, que estaba hecha con funciones, y la reconstruye con clases conectadas por un `Pipeline`. Además, comprueba que ambas versiones entregan exactamente el mismo resultado. El propósito es dejar el código organizado en piezas que se puedan reutilizar, probar y ampliar.

## Cómo se integra con las fases anteriores

- **Entrada:** `data/processed/egresos_araucania_2022.csv`, generado en F2_01.
- **Código reutilizado:** las funciones de `src/transformacion.py` (Fase 2), sin duplicarlas.
- **Código nuevo de F3:** `src/base.py`, `src/pipeline.py`, `src/etapas.py` y `src/proporciones.py`.
- **Notebook complementario:** `F3_02_eficiencia.ipynb`, con las mediciones de tiempo y memoria.


In [1]:
import sys
from pathlib import Path

import pandas as pd

# El notebook se ejecuta desde F3/; la raíz del repositorio es la carpeta superior
RAIZ = Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.entorno import verificar_entorno
from src import transformacion as tr
from src.pipeline import Pipeline
from src.etapas import (
    FiltroColelitiasis,
    AsignadorProvincia,
    MarcadorGruposRiesgo,
    MarcadorColecistectomia,
)

versiones = verificar_entorno()

Versión de Python: 3.13.17
Entorno virtual del proyecto (.venv): True
pandas: 3.1.1
numpy: 2.5.4
matplotlib: 3.11.2
seaborn: 0.13.2


## 1. Carga de los egresos de La Araucanía

Se lee la salida de F2_01: los egresos hospitalarios de 2022 de residentes de La Araucanía. Todas las columnas se leen como texto, igual que en la Fase 2, para conservar los códigos tal como vienen (por ejemplo, el cero inicial de las comunas).

In [2]:
ENTRADA = RAIZ / "data" / "processed" / "egresos_araucania_2022.csv"

araucania = pd.read_csv(ENTRADA, dtype=str)
print("Filas y columnas:", araucania.shape)
araucania.head(3)

Filas y columnas: (84618, 18)


,PERTENENCIA_ESTABLECIMIENTO_SALUD,SEXO,GRUPO_EDAD,ETNIA,GLOSA_PAIS_ORIGEN,COMUNA_RESIDENCIA,GLOSA_COMUNA_RESIDENCIA,REGION_RESIDENCIA,GLOSA_REGION_RESIDENCIA,PREVISION,GLOSA_PREVISION,ANO_EGRESO,DIAG1,DIAG2,DIAS_ESTADA,CONDICION_EGRESO,GLOSA_INTERV_Q_PPAL,GLOSA_PROCED_PPAL
0,No Pertenecientes al Sistema Nacional de Servi...,HOMBRE,1 a 9,No se identifica con alguna etnia,Chileno,09101,Temuco,09,De La Araucanía,1,FONASA,2022,J353,NaN,1,1,"Amigdalectomía c/s adenoidectomía, uni o bilat...",NaN
1,No Pertenecientes al Sistema Nacional de Servi...,HOMBRE,1 a 9,No se identifica con alguna etnia,Chileno,09101,Temuco,09,De La Araucanía,1,FONASA,2022,J353,NaN,1,1,"Amigdalectomía c/s adenoidectomía, uni o bilat...",NaN
2,No Pertenecientes al Sistema Nacional de Servi...,HOMBRE,1 a 9,No se identifica con alguna etnia,Chileno,09202,Collipulli,09,De La Araucanía,1,FONASA,2022,Q531,NaN,1,1,Descenso testículo inguinal c/s hernioplastía,NaN


## 2. Versión estructurada (Fase 2)

En la Fase 2 la transformación se resolvió con cuatro funciones encadenadas dentro de `transformar()`: filtrar colelitiasis, asignar provincia, marcar grupos de riesgo y marcar colecistectomía. Este resultado sirve como referencia para verificar la versión orientada a objetos.

In [3]:
referencia = tr.transformar(araucania)
print("Egresos por colelitiasis:", len(referencia))
print("Columnas:", referencia.shape[1])

Egresos por colelitiasis: 3789
Columnas: 23


## 3. Versión orientada a objetos (Fase 3)

Cada paso de la transformación es ahora una clase que hereda de `EtapaBase` e implementa el método `ejecutar`. Primero se ejecutan las etapas una por una para observar los resultados intermedios; después se conectan con el `Pipeline`.

### 3.1 Etapas por separado: herencia y polimorfismo

In [4]:
etapas = [
    FiltroColelitiasis(),
    AsignadorProvincia(),
    MarcadorGruposRiesgo(),
    MarcadorColecistectomia(),
]

# Polimorfismo: el mismo mensaje (ejecutar) se envía a objetos de clases distintas
datos = araucania
for etapa in etapas:
    datos = etapa.ejecutar(datos)
    clase = type(etapa).__name__
    madre = type(etapa).__bases__[0].__name__
    print(f"{clase:<24} hereda de {madre} -> {datos.shape[0]:>6} filas, {datos.shape[1]} columnas")

FiltroColelitiasis       hereda de EtapaBase ->   3789 filas, 18 columnas
AsignadorProvincia       hereda de EtapaBase ->   3789 filas, 19 columnas
MarcadorGruposRiesgo     hereda de EtapaBase ->   3789 filas, 22 columnas
MarcadorColecistectomia  hereda de EtapaBase ->   3789 filas, 23 columnas


### 3.2 Etapas conectadas con el Pipeline

El `Pipeline` recibe la lista de etapas y entrega la salida de cada una como entrada de la siguiente. Guarda las etapas en un atributo interno (`_etapas`) y solo permite consultarlas mediante la propiedad `etapas` (encapsulamiento).

In [5]:
flujo = Pipeline(etapas)
resultado = flujo.ejecutar(araucania)

print("Etapas configuradas:", [type(e).__name__ for e in flujo.etapas])
print("Egresos por colelitiasis:", len(resultado))
print("Columnas:", resultado.shape[1])

Etapas configuradas: ['FiltroColelitiasis', 'AsignadorProvincia', 'MarcadorGruposRiesgo', 'MarcadorColecistectomia']
Egresos por colelitiasis: 3789
Columnas: 23


## 4. Verificación

Se comprueba que la versión orientada a objetos entrega el mismo resultado que la versión estructurada y que el archivo generado en F2_03. Luego se prueban un caso límite y dos excepciones.

In [6]:
# 1) La versión con clases entrega lo mismo que la versión con funciones
assert resultado.equals(referencia)
print("OK  Pipeline (F3) == transformar() (F2)")

# 2) Coincide con el archivo generado en F2_03
CSV_F2 = RAIZ / "data" / "processed" / "egresos_araucania_k80_transformado.csv"
csv_f2 = pd.read_csv(CSV_F2, dtype=str)
nuevas = [
    "PROVINCIA_RESIDENCIA",
    "GRUPO_RIESGO_MUJER",
    "GRUPO_RIESGO_ETNIA",
    "GRUPO_RIESGO_40_49",
    "FLAG_COLECISTECTOMIA",
]
comparable = resultado[nuevas].astype(str).reset_index(drop=True)
assert len(resultado) == len(csv_f2)
assert (comparable.values == csv_f2[nuevas].values).all()
print(f"OK  {len(resultado)} egresos y 5 columnas derivadas idénticas al CSV de F2_03")

OK  Pipeline (F3) == transformar() (F2)
OK  3789 egresos y 5 columnas derivadas idénticas al CSV de F2_03


In [7]:
# Caso límite: una tabla sin filas no debe producir error
vacio = araucania.iloc[0:0]
print("OK  tabla vacía ->", flujo.ejecutar(vacio).shape)

# Excepción 1: falta una columna necesaria
try:
    flujo.ejecutar(araucania.drop(columns=["DIAG1"]))
except KeyError as error:
    print("OK  columna faltante -> KeyError:", error)

# Excepción 2: un Pipeline sin etapas no tiene sentido
try:
    Pipeline([])
except ValueError as error:
    print("OK  sin etapas -> ValueError:", error)

OK  tabla vacía -> (0, 23)
OK  columna faltante -> KeyError: 'filtrar_colelitiasis: falta la columna DIAG1.'
OK  sin etapas -> ValueError: Pipeline: se requiere al menos una etapa.


In [8]:
# Caso límite: una tabla sin filas no debe producir error
vacio = araucania.iloc[0:0]
print("OK  tabla vacía ->", flujo.ejecutar(vacio).shape)

# Excepción 1: falta una columna necesaria
try:
    flujo.ejecutar(araucania.drop(columns=["DIAG1"]))
except KeyError as error:
    print("OK  columna faltante -> KeyError:", error)

# Excepción 2: un Pipeline sin etapas no tiene sentido
try:
    Pipeline([])
except ValueError as error:
    print("OK  sin etapas -> ValueError:", error)

OK  tabla vacía -> (0, 23)
OK  columna faltante -> KeyError: 'filtrar_colelitiasis: falta la columna DIAG1.'
OK  sin etapas -> ValueError: Pipeline: se requiere al menos una etapa.


## 5. Versión recursiva del flujo

El `Pipeline` recorre sus etapas con un bucle. El mismo flujo se puede expresar de forma recursiva: ejecutar una lista de etapas equivale a ejecutar la primera y luego, recursivamente, el resto.

- **Caso base:** si no quedan etapas, se devuelven los datos.
- **Paso recursivo:** se ejecuta la primera etapa y la función se llama a sí misma con las etapas restantes.

In [9]:
def ejecutar_recursivo(etapas, datos):
    """Ejecuta las etapas en orden mediante recursividad."""
    if not etapas:                      # caso base: no quedan etapas
        return datos
    primera, *resto = etapas            # separa la primera etapa del resto
    return ejecutar_recursivo(resto, primera.ejecutar(datos))  # paso recursivo


resultado_recursivo = ejecutar_recursivo(etapas, araucania)
assert resultado_recursivo.equals(resultado)
print("OK  versión recursiva == versión con bucle (Pipeline)")

OK  versión recursiva == versión con bucle (Pipeline)


In [10]:
import timeit

REPETICIONES = 5
LLAMADAS = 3

versiones_flujo = {
    "Bucle (Pipeline)": lambda: flujo.ejecutar(araucania),
    "Recursiva": lambda: ejecutar_recursivo(etapas, araucania),
}

for nombre, funcion in versiones_flujo.items():
    tiempos = timeit.repeat(funcion, number=LLAMADAS, repeat=REPETICIONES)
    print(f"{nombre:<18} {min(tiempos) / LLAMADAS * 1000:6.1f} ms por ejecución (mínimo de {REPETICIONES})")

Bucle (Pipeline)     20.1 ms por ejecución (mínimo de 5)
Recursiva            11.5 ms por ejecución (mínimo de 5)


### Interpretación

- **Resultado:** ambas versiones entregan exactamente la misma tabla.
- **Tiempo:** la versión recursiva fue más rápida (cerca de 39 ms frente a 70 ms), pero la diferencia no se debe a la recursividad: el `Pipeline` hace una copia de protección de los datos de entrada y valida el resultado de cada etapa, tareas que la función recursiva no realiza.
- **Complejidad temporal:** las dos recorren cada etapa una vez. Con *k* etapas y *n* egresos, ambas son O(k·n).
- **Complejidad espacial:** la versión recursiva apila una llamada por etapa (profundidad 4, muy lejos del límite de Python, cercano a 1.000 llamadas). El bucle no apila llamadas.
- **Elección:** se mantiene el `Pipeline` con bucle, porque es más claro, protege los datos de entrada, valida cada paso y no tiene límite de profundidad. Una diferencia de unos 30 ms no justifica perder esas garantías.

## 6. Cálculo de proporciones

La tabla que entrega el `Pipeline` se usa como entrada de la clase de proporciones (`src/proporciones.py`), que calcula la proporción de egresos con colecistectomía por grupo de riesgo y provincia. Se usa la estrategia `CalcNumpy`, elegida en `F3_02_eficiencia.ipynb` por ser la más rápida.

In [11]:
from src.proporciones import Columnas, CalcNumpy

columnas = Columnas(
    provincia="PROVINCIA_RESIDENCIA",
    flag="FLAG_COLECISTECTOMIA",
    grupos=("GRUPO_RIESGO_MUJER", "GRUPO_RIESGO_ETNIA", "GRUPO_RIESGO_40_49"),
)

calculador = CalcNumpy(resultado, columnas)
print(f"Proporción global con colecistectomía: {calculador.proporcion_global():.1%}")

tabla = calculador.calcular()
en_riesgo = tabla[tabla["en_grupo"]].copy()
en_riesgo["proporcion_%"] = (en_riesgo["proporcion"] * 100).round(1)
en_riesgo[["grupo", "provincia", "egresos", "con_colecistectomia", "proporcion_%"]]

Proporción global con colecistectomía: 80.7%


,grupo,provincia,egresos,con_colecistectomia,proporcion_%
1,GRUPO_RIESGO_40_49,Cautín,521,465,89.3
3,GRUPO_RIESGO_40_49,Malleco,127,100,78.7
5,GRUPO_RIESGO_ETNIA,Cautín,633,499,78.8
7,GRUPO_RIESGO_ETNIA,Malleco,120,85,70.8
9,GRUPO_RIESGO_MUJER,Cautín,2211,1844,83.4
11,GRUPO_RIESGO_MUJER,Malleco,553,433,78.3


## 7. Cierre y trazabilidad

| Elemento | Archivo | Qué aporta |
|---|---|---|
| Funciones de transformación (F2) | `src/transformacion.py` | Lógica de filtro, provincia, grupos de riesgo y colecistectomía |
| Clase base y flujo | `src/base.py`, `src/pipeline.py` | Contrato común (`EtapaBase`) y ejecución ordenada (`Pipeline`) |
| Etapas concretas | `src/etapas.py` | Cuatro clases que heredan de `EtapaBase` y reutilizan las funciones de F2 |
| Proporciones | `src/proporciones.py` | Tres estrategias de cálculo intercambiables |
| Eficiencia | `F3/F3_02_eficiencia.ipynb` | Mediciones con `timeit` y `tracemalloc` |
| Arquitectura | `docs/arquitectura_f3.md` | Diagrama de clases y patrón de diseño |
| Pruebas | `tests/test_pipeline.py` | Nueve pruebas automatizadas del flujo |

**Qué se comprobó en este notebook:** la versión orientada a objetos entrega el mismo resultado que la versión estructurada de la Fase 2 y que el archivo generado en F2_03 (3.789 egresos); el flujo responde correctamente ante una tabla vacía y ante entradas inválidas; y la versión recursiva es equivalente a la versión con bucle.

### Lectura de los resultados

En los tres grupos de riesgo, Malleco presenta una menor proporción de colecistectomía que Cautín. La mayor diferencia está en el grupo de 40 a 49 años (89,3 % en Cautín frente a 78,7 % en Malleco), y el valor más bajo corresponde a las personas que se identifican con una etnia en Malleco (70,8 %).

Estos resultados son descriptivos. Los grupos de Malleco son pequeños (120 y 127 egresos en dos de ellos) y, sin intervalos de confianza, no es posible afirmar que las diferencias sean estadísticamente significativas ni explicar su causa.